# Cuaderno 5 · Modelado y validación con incendios reales (v2)

**Objetivo:** entrenar modelos que estimen la **probabilidad de que una zona con vegetación arda**, aprendiendo de los incendios observados en Canarias, y evaluarlos de forma que el resultado sea creíble.

Este cuaderno responde directamente a los puntos de mejora del tribunal del TFM:

| Punto del tribunal | Dónde se resuelve |
|---|---|
| 1. Separar espacial y temporalmente entrenamiento y prueba | Secciones 3 y 4: validación por islas y bloques de 10 km, y prueba con años posteriores |
| 2. Contraste externo con incendios posteriores y calibración | Secciones 5 y 6: prueba 2024–2026 y calibración isotónica |
| 3. Sensibilidad, precisión y calibración en zonas de alto riesgo | Secciones 5 y 6: sensibilidad, falsos positivos, captura en el 10 % / 20 % de mayor riesgo y diagrama de fiabilidad |

Además, se compara con el **índice heurístico de la v1 (TFM)** como modelo de referencia: si los modelos aprendidos no lo superan, no aportan valor.

**Entrada:** `dataset_v2.csv`, generado por el `Cuaderno4`.

## 1. Entorno

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, precision_recall_curve

warnings.filterwarnings("ignore", category=UserWarning)

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # Clono el repositorio para reutilizar el paquete `wildfire` (src/wildfire)
    if not os.path.exists("/content/repo"):
        !git clone -q https://github.com/joseandres03/sentinel2-meteo_IA_wildfire_risk.git /content/repo
    sys.path.insert(0, "/content/repo/src")
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_TRABAJO = "/content/drive/MyDrive/TFM_Riesgo_Incendios_Canarias/v2"
else:
    sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
    RUTA_TRABAJO = os.environ.get("RUTA_TRABAJO_V2", "datos_v2")

from wildfire import indices, validacion

RUTA_DATASET = os.path.join(RUTA_TRABAJO, "dataset_v2.csv")
RUTA_SALIDA = os.path.join(RUTA_TRABAJO, "modelos")
os.makedirs(RUTA_SALIDA, exist_ok=True)

FECHA_CORTE_PRUEBA = "2024-01-01"   # entreno con 2017-2023 y pruebo con 2024 en adelante
SEMILLA = 42

## 2. Datos y variables

Cada fila es un **punto con vegetación en una fecha**. La etiqueta `incendio` vale 1 si ese día empezó un incendio en ese punto (FIRMS) y 0 si no hubo ninguno cerca en espacio y tiempo.

Todas las variables describen la situación **antes** del incendio: la imagen Sentinel-2 es una mediana sin nubes de los 30 días previos, y la meteorología es la del propio día (en producción se usaría el pronóstico).

In [ ]:
df = pd.read_csv(RUTA_DATASET, parse_dates=["fecha"])
print(f"{len(df)} ejemplos · {df['incendio'].sum()} incendios · prevalencia {df['incendio'].mean():.3f}")

# Variables de calendario (la estacionalidad es un factor real del riesgo)
df["mes_sin"] = np.sin(2 * np.pi * df["fecha"].dt.month / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["fecha"].dt.month / 12)

VARIABLES = [
    # Estado de la vegetación (Sentinel-2)
    "ndvi", "ndmi", "nbr", "B2", "B3", "B4", "B8", "B11", "B12",
    # Meteorología (ERA5-Land)
    "tmax_c", "hr_media", "viento_ms", "precip_30d_mm",
    # Relieve y cobertura del suelo
    "elevation", "slope", "orientacion_norte",
    "frac_arbolado", "frac_matorral", "frac_pastizal", "frac_cultivo", "frac_urbano",
    # Calendario
    "mes_sin", "mes_cos",
]
VARIABLES = [v for v in VARIABLES if v in df.columns]

print("\nValores ausentes por variable (%):")
print((df[VARIABLES].isna().mean() * 100).round(1).sort_values(ascending=False).head(8))

resumen = df.groupby("isla")["incendio"].agg(ejemplos="size", incendios="sum")
resumen["prevalencia"] = (resumen["incendios"] / resumen["ejemplos"]).round(3)
resumen

### Índice de la v1 como referencia

Recalculo el índice del TFM (`0,35 × meteo + 0,65 × combustible`) con las variables disponibles. El riesgo de combustible se aproxima con el NDMI medio del entorno; en la v1 se promediaba sobre los píxeles de vegetación del parche.

In [ ]:
riesgo_meteo_v1 = indices.riesgo_meteorologico_v1(df["tmax_c"], df["hr_media"], df["viento_ms"])
riesgo_comb_v1 = np.clip((1 - df["ndmi"]) / 2, 0, 1).fillna(0)
df["indice_v1"] = indices.indice_v1(riesgo_meteo_v1, riesgo_comb_v1)
df[["indice_v1", "incendio"]].groupby("incendio").describe().round(3)

## 3. Partición temporal

- **Entrenamiento:** incendios y no incendios anteriores a 2024.
- **Prueba:** 2024 en adelante. Es el **contraste externo con incendios posteriores** que pedía el tribunal: el modelo nunca ha visto estos años.

In [ ]:
ent, prueba = validacion.particion_temporal(df["fecha"], FECHA_CORTE_PRUEBA)
df_ent, df_prueba = df.loc[ent].reset_index(drop=True), df.loc[prueba].reset_index(drop=True)
X_ent, y_ent = df_ent[VARIABLES], df_ent["incendio"].to_numpy()
X_prueba, y_prueba = df_prueba[VARIABLES], df_prueba["incendio"].to_numpy()

print(f"Entrenamiento: {len(df_ent)} ejemplos, {y_ent.sum()} incendios ({df_ent['fecha'].min():%Y}-{df_ent['fecha'].max():%Y})")
print(f"Prueba:        {len(df_prueba)} ejemplos, {y_prueba.sum()} incendios ({df_prueba['fecha'].min():%Y}-{df_prueba['fecha'].max():%Y})")

## 4. Modelos y validación cruzada espacial

In [ ]:
def crear_modelos():
    modelos = {
        "Regresión logística": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced")),
        "Random Forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(n_estimators=400, min_samples_leaf=5, class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=SEMILLA)),
    }
    try:
        from xgboost import XGBClassifier
        peso = (y_ent == 0).sum() / max(1, (y_ent == 1).sum())
        modelos["XGBoost"] = XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=peso, eval_metric="aucpr", random_state=SEMILLA, n_jobs=-1)
    except ImportError:
        modelos["Gradient Boosting"] = HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=SEMILLA)
    return modelos

modelos = crear_modelos()
list(modelos)

### ¿Cuánto infla la validación aleatoria los resultados?

Comparo tres formas de validar sobre el periodo de entrenamiento:

1. **K-fold aleatorio** (lo que se hizo en la v1): puntos vecinos pueden caer unos en entrenamiento y otros en validación.
2. **Bloques espaciales de 10 km**: los puntos de un mismo bloque van siempre juntos.
3. **Dejando una isla fuera**: el modelo se evalúa en una isla que no ha visto.

Si el AUC baja mucho al pasar de 1 a 2 o 3, el modelo estaba aprovechando la **autocorrelación espacial**.

In [ ]:
grupos_bloque = validacion.bloques_espaciales(df_ent["lat"], df_ent["lon"], tam_km=10)
grupos_isla = df_ent["isla"].to_numpy()

def oof_aleatorio(modelo, X, y, n_splits=5):
    from sklearn.base import clone
    oof = np.zeros(len(y))
    for i_ent, i_val in StratifiedKFold(n_splits, shuffle=True, random_state=SEMILLA).split(X, y):
        m = clone(modelo).fit(X.iloc[i_ent], y[i_ent])
        oof[i_val] = m.predict_proba(X.iloc[i_val])[:, 1]
    return oof

filas = []
for nombre, modelo in modelos.items():
    esquemas = {
        "K-fold aleatorio": oof_aleatorio(modelo, X_ent, y_ent),
        "Bloques 10 km": validacion.prediccion_fuera_de_bloque(modelo, X_ent, y_ent, grupos_bloque),
        "Isla fuera": validacion.prediccion_fuera_de_bloque(modelo, X_ent, y_ent, grupos_isla, n_splits=7),
    }
    for esquema, oof in esquemas.items():
        m = validacion.metricas_clasificacion(y_ent, oof)
        filas.append({"modelo": nombre, "validación": esquema, "ROC-AUC": m["roc_auc"],
                      "PR-AUC": m["pr_auc"], "captura top 10 %": m["captura_top10"]})

m_v1 = validacion.metricas_clasificacion(y_ent, df_ent["indice_v1"])
filas.append({"modelo": "Índice v1 (TFM)", "validación": "sin entrenamiento", "ROC-AUC": m_v1["roc_auc"],
              "PR-AUC": m_v1["pr_auc"], "captura top 10 %": m_v1["captura_top10"]})

tabla_cv = pd.DataFrame(filas)
tabla_cv.pivot_table(index="modelo", columns="validación", values="ROC-AUC").round(3)

In [ ]:
tabla_cv.round(3)

## 5. Evaluación final con incendios posteriores (2024 en adelante)

Entreno cada modelo con todo el periodo 2017–2023 y lo evalúo con los años de prueba.

- **ROC-AUC:** capacidad de ordenar zonas por riesgo (0,5 = azar).
- **PR-AUC:** más exigente con clases desbalanceadas; el azar equivale a la prevalencia.
- **Captura top 10 % / 20 %:** fracción de incendios reales que caen en el 10 % / 20 % de casos con más riesgo predicho.
- **Sensibilidad, precisión, falsos positivos e incendios omitidos** con umbral 0,5.

In [ ]:
resultados, predicciones = [], {}
for nombre, modelo in modelos.items():
    modelo.fit(X_ent, y_ent)
    p = modelo.predict_proba(X_prueba)[:, 1]
    predicciones[nombre] = p
    resultados.append({"modelo": nombre, **validacion.metricas_clasificacion(y_prueba, p)})

predicciones["Índice v1 (TFM)"] = df_prueba["indice_v1"].to_numpy()
resultados.append({"modelo": "Índice v1 (TFM)",
                   **validacion.metricas_clasificacion(y_prueba, predicciones["Índice v1 (TFM)"])})

tabla_prueba = pd.DataFrame(resultados).set_index("modelo")
tabla_prueba[["roc_auc", "pr_auc", "captura_top10", "captura_top20", "sensibilidad",
              "precision", "falsos_positivos", "incendios_omitidos", "brier"]].round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
for nombre, p in predicciones.items():
    fpr, tpr, _ = roc_curve(y_prueba, p)
    prec, rec, _ = precision_recall_curve(y_prueba, p)
    ejes[0].plot(fpr, tpr, label=f"{nombre} (AUC {tabla_prueba.loc[nombre, 'roc_auc']:.2f})")
    ejes[1].plot(rec, prec, label=f"{nombre} (AP {tabla_prueba.loc[nombre, 'pr_auc']:.2f})")
ejes[0].plot([0, 1], [0, 1], "k--", lw=1)
ejes[1].axhline(y_prueba.mean(), color="k", ls="--", lw=1, label="azar")
ejes[0].set(title="Curva ROC (prueba 2024+)", xlabel="Tasa de falsos positivos", ylabel="Sensibilidad")
ejes[1].set(title="Curva precisión-sensibilidad (prueba 2024+)", xlabel="Sensibilidad", ylabel="Precisión")
for ax in ejes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, "curvas_roc_pr.png"), dpi=150)
plt.show()

## 6. Calibración de probabilidades

Que un modelo ordene bien no significa que sus probabilidades sean fiables: un 70 % debería cumplirse aproximadamente 7 de cada 10 veces. Para comprobarlo:

1. Elijo el mejor modelo por PR-AUC en la validación espacial (no en la prueba, para no hacer trampa).
2. Lo calibro con **regresión isotónica** usando validación cruzada por bloques espaciales.
3. Comparo la **tabla de fiabilidad** antes y después en los años de prueba.

In [ ]:
cv_bloques = tabla_cv[(tabla_cv["validación"] == "Bloques 10 km")]
mejor = cv_bloques.sort_values("PR-AUC", ascending=False)["modelo"].iloc[0]
print("Modelo elegido por validación espacial:", mejor)

modelo_base = crear_modelos()[mejor]
particiones = list(GroupKFold(n_splits=5).split(X_ent, y_ent, grupos_bloque))
calibrado = CalibratedClassifierCV(modelo_base, method="isotonic", cv=particiones)
calibrado.fit(X_ent, y_ent)
p_cal = calibrado.predict_proba(X_prueba)[:, 1]

antes = validacion.tabla_fiabilidad(y_prueba, predicciones[mejor])
despues = validacion.tabla_fiabilidad(y_prueba, p_cal)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="calibración perfecta")
ax.plot(antes["prob_media"], antes["frecuencia_observada"], "o-", label="sin calibrar")
ax.plot(despues["prob_media"], despues["frecuencia_observada"], "s-", label="calibrado (isotónica)")
ax.set(xlabel="Probabilidad predicha", ylabel="Frecuencia observada de incendio",
       title=f"Diagrama de fiabilidad · {mejor} · prueba 2024+")
ax.legend()
plt.savefig(os.path.join(RUTA_SALIDA, "fiabilidad.png"), dpi=150)
plt.show()

m_cal = validacion.metricas_clasificacion(y_prueba, p_cal)
print(f"Brier sin calibrar: {tabla_prueba.loc[mejor, 'brier']:.4f} · calibrado: {m_cal['brier']:.4f}")
despues

### De probabilidad en la muestra a probabilidad real

En el dataset hay aproximadamente 1 incendio por cada 10 negativos, pero en la realidad un punto concreto arde en un día concreto con una probabilidad muchísimo menor. Las probabilidades calibradas lo son **respecto a la muestra**. Para el visor se pueden:

- usar como **índice relativo** (ordenar zonas), que es lo que mide bien el ROC-AUC, o
- convertir a probabilidad real con la corrección de prevalencia (`validacion.corregir_prevalencia`), estimando la prevalencia real como *eventos / (puntos con vegetación × días)*.

In [ ]:
prev_muestra = y_ent.mean()
celdas_vegetacion_1km = 3000          # orden de magnitud de celdas de 1 km² con vegetación en Canarias (ajustar)
dias = (df_ent["fecha"].max() - df_ent["fecha"].min()).days
prev_real = y_ent.sum() / (celdas_vegetacion_1km * dias)
print(f"Prevalencia en la muestra: {prev_muestra:.3f} · prevalencia real estimada: {prev_real:.2e}")

ejemplo = np.array([0.1, 0.3, 0.5, 0.8])
pd.DataFrame({"prob. calibrada (muestra)": ejemplo,
              "prob. real diaria por km²": validacion.corregir_prevalencia(ejemplo, prev_muestra, prev_real)})

## 7. ¿Qué variables usa el modelo?

Importancia por permutación sobre los años de prueba: cuánto empeora el PR-AUC si se desordena cada variable.

In [ ]:
modelo_final = modelos[mejor]
imp = permutation_importance(modelo_final, X_prueba, y_prueba, scoring="average_precision",
                             n_repeats=10, random_state=SEMILLA, n_jobs=-1)
importancias = pd.Series(imp.importances_mean, index=VARIABLES).sort_values()
importancias.tail(15).plot.barh(figsize=(7, 6), title=f"Importancia por permutación · {mejor}")
plt.xlabel("Descenso del PR-AUC")
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, "importancia_variables.png"), dpi=150)
plt.show()

## 8. Guardado del modelo y de las métricas

In [ ]:
import joblib

joblib.dump({"modelo": calibrado, "variables": VARIABLES, "modelo_base": mejor,
             "fecha_corte_prueba": FECHA_CORTE_PRUEBA, "prevalencia_muestra": float(prev_muestra)},
            os.path.join(RUTA_SALIDA, "modelo_v2_calibrado.joblib"))

informe = {
    "fecha_corte_prueba": FECHA_CORTE_PRUEBA,
    "n_entrenamiento": int(len(df_ent)), "n_prueba": int(len(df_prueba)),
    "incendios_entrenamiento": int(y_ent.sum()), "incendios_prueba": int(y_prueba.sum()),
    "modelo_elegido": mejor,
    "validacion_cruzada": tabla_cv.round(4).to_dict(orient="records"),
    "prueba_temporal": tabla_prueba.round(4).reset_index().to_dict(orient="records"),
    "prueba_temporal_calibrado": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m_cal.items()},
}
with open(os.path.join(RUTA_SALIDA, "metricas_v2.json"), "w", encoding="utf-8") as f:
    json.dump(informe, f, indent=2, ensure_ascii=False)
print("Guardado en", RUTA_SALIDA)

## 9. Conclusiones

> Rellenar tras ejecutar el cuaderno con los datos reales. Puntos a comentar:
>
> - Diferencia de AUC entre la validación aleatoria y la espacial (tamaño de la fuga de la v1).
> - Si los modelos entrenados superan al índice v1 en los años de prueba y por cuánto.
> - Captura de incendios en el 10 % / 20 % de zonas de mayor riesgo y número de falsos positivos.
> - Calidad de la calibración y si la probabilidad puede mostrarse como tal en el visor.
> - Variables más influyentes y si tienen sentido físico.
> - Limitaciones: resolución de 1 km de FIRMS (incendios pequeños no detectados), quemas agrícolas, número de eventos por isla.